# Reference-embedding random forest: minimal worked example

This notebook uses the Tellus CSV to choose a training design quickly in scikit-learn, validates the corresponding Earth Engine random forest, saves both the observations-plus-reference-year embeddings and the final Earth Engine classifier as assets, and writes one JSON report containing the data behind every figure. It trains the reference model; prediction-year imagery is outside this notebook.

In [ ]:
!pip -q install earthengine-api pandas scikit-learn matplotlib

import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from google.colab import userdata

ee.Authenticate()
EE_PROJECT = userdata.get('cloud_project')
ee.Initialize(project=EE_PROJECT)

## Configuration

In [ ]:
CSV_NAME = 'tellus_shallow_topsoil_A_loi.csv'
RUN_NAME = 'tellus_loi_2018_consolidated'  # must be new for each run

LON, LAT = 'lon', 'lat'
X_METRES, Y_METRES = 'X_ITM', 'Y_ITM'
TARGET_COLUMN, TARGET_THRESHOLD = 'LOI_PCT', 30.0
REFERENCE_YEAR = 2018
BLOCK_SIZE_M = 10_000
TEST_BLOCK_FRACTION = 0.20
POINTS_PER_BLOCK = [1, 2, 5, 10, 20, 'all']
BLOCK_FRACTIONS = [0.10, 0.25, 0.50, 0.75, 1.00]
AUC_TOLERANCE = 0.01
SEED = 42

N_TREES = 100
N_BANDS = 64
VARS_PER_SPLIT = int(np.sqrt(N_BANDS))
MIN_LEAF_POPULATION = 1
GEE_BAG_FRACTION = 0.5
EMBEDDINGS = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
BANDS = [f'A{i:02d}' for i in range(N_BANDS)]
SCALE_M = 10

POINT_ASSET = f'projects/{EE_PROJECT}/assets/{RUN_NAME}_points'
SAMPLE_ASSET = f'projects/{EE_PROJECT}/assets/{RUN_NAME}_reference_samples'
MODEL_ASSET = f'projects/{EE_PROJECT}/assets/{RUN_NAME}_random_forest'

## Upload the observations

In [ ]:
import time

existing = []
for asset_id in [POINT_ASSET, SAMPLE_ASSET, MODEL_ASSET]:
    try:
        ee.data.getAsset(asset_id)
        existing.append(asset_id)
    except ee.EEException:
        pass
assert not existing, f'Change RUN_NAME; these assets already exist: {existing}'

files.upload()
observations = pd.read_csv(CSV_NAME)
required = [LON, LAT, X_METRES, Y_METRES, TARGET_COLUMN]
missing = [name for name in required if name not in observations]
assert not missing, f'Missing columns: {missing}'
observations = observations.dropna(subset=required).reset_index(drop=True)
observations['_row'] = np.arange(len(observations))
observations['y'] = (observations[TARGET_COLUMN] >= TARGET_THRESHOLD).astype(int)
observations['block_id'] = (
    (observations[X_METRES] // BLOCK_SIZE_M).astype(int).astype(str) + '_' +
    (observations[Y_METRES] // BLOCK_SIZE_M).astype(int).astype(str))

def wait(task):
    while task.active():
        time.sleep(5)
    status = task.status()
    if status['state'] != 'COMPLETED':
        raise RuntimeError(status.get('error_message', status))

def wait_for_asset(asset_id):
    for _ in range(30):
        try:
            ee.data.getAsset(asset_id)
            return
        except ee.EEException:
            time.sleep(2)
    raise RuntimeError(f'Asset was created but is not readable: {asset_id}')

features = [
    ee.Feature(ee.Geometry.Point([float(lon), float(lat)]),
               {'_row': int(row), 'lon': float(lon), 'lat': float(lat),
                'y': int(y), 'block_id': str(block)})
    for row, lon, lat, y, block in observations[
        ['_row', LON, LAT, 'y', 'block_id']].itertuples(index=False, name=None)]
task = ee.batch.Export.table.toAsset(
    ee.FeatureCollection(features), f'{RUN_NAME}_points', POINT_ASSET)
task.start(); wait(task); wait_for_asset(POINT_ASSET)
points = ee.FeatureCollection(POINT_ASSET)
print(f'{len(observations):,} observations saved to {POINT_ASSET}')

## Sample and save the reference-year embeddings

In [ ]:
def embedding(year):
    return (ee.ImageCollection(EMBEDDINGS)
            .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
            .filterBounds(points.geometry())
            .mosaic().select(BANDS))

reference_samples = embedding(REFERENCE_YEAR).sampleRegions(
    collection=points,
    properties=['_row', 'lon', 'lat', 'y', 'block_id'],
    scale=SCALE_M, geometries=True, tileScale=4)
task = ee.batch.Export.table.toAsset(
    reference_samples, f'{RUN_NAME}_reference_samples', SAMPLE_ASSET)
task.start(); wait(task); wait_for_asset(SAMPLE_ASSET)
reference_samples = ee.FeatureCollection(SAMPLE_ASSET)
reference = ee.data.computeFeatures({
    'expression': reference_samples,
    'fileFormat': 'PANDAS_DATAFRAME',
    'pageSize': 5000})
reference = reference.dropna(subset=BANDS).reset_index(drop=True)
reference['_row'] = reference['_row'].astype(int)
reference['y'] = reference['y'].astype(int)
print(f'{len(reference):,} observations with embeddings saved to {SAMPLE_ASSET}')

## Spatial holdout

In [ ]:
rng = np.random.default_rng(SEED)
blocks = np.sort(reference.block_id.unique())
test_blocks = set(rng.choice(
    blocks, size=round(TEST_BLOCK_FRACTION * len(blocks)), replace=False))
train = reference[~reference.block_id.isin(test_blocks)].copy()
test = reference[reference.block_id.isin(test_blocks)].copy()
assert train.y.nunique() == 2 and test.y.nunique() == 2

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(train.lon, train.lat, s=3, alpha=.35, label=f'Train: {train.block_id.nunique()} blocks')
ax.scatter(test.lon, test.lat, s=3, alpha=.55, label=f'Holdout: {test.block_id.nunique()} blocks')
ax.set(xlabel='Longitude', ylabel='Latitude', title='Training and held-out observations')
ax.legend(frameon=False); plt.show()
print(f'{len(train):,} training observations; {len(test):,} held out')

## Fast training-design experiments in scikit-learn

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

train = train.assign(_order=np.random.default_rng(SEED).random(len(train)))
block_order = np.random.default_rng(SEED).permutation(train.block_id.unique())

def cap_per_block(frame, cap):
    ordered = frame.sort_values(['block_id', '_order'])
    return ordered if cap == 'all' else ordered.groupby('block_id').head(cap)

def fit_score(frame):
    model = RandomForestClassifier(
        n_estimators=N_TREES, max_features='sqrt',
        random_state=SEED, n_jobs=-1)
    model.fit(frame[BANDS], frame.y)
    probability = model.predict_proba(test[BANDS])[:, 1]
    return model, probability, {
        'n_points': int(len(frame)), 'n_blocks': int(frame.block_id.nunique()),
        'auc': roc_auc_score(test.y, probability),
        'accuracy': accuracy_score(test.y, probability >= .5)}

point_results = []
for cap in POINTS_PER_BLOCK:
    _, _, result = fit_score(cap_per_block(train, cap))
    point_results.append({'points_per_block': cap, **result})

full_auc = point_results[-1]['auc']
selected_cap = next(row['points_per_block'] for row in point_results
                    if row['auc'] >= full_auc - AUC_TOLERANCE)
capped_train = cap_per_block(train, selected_cap)

block_results = []
for fraction in BLOCK_FRACTIONS:
    n = max(1, round(fraction * len(block_order)))
    subset = capped_train[capped_train.block_id.isin(block_order[:n])]
    _, _, result = fit_score(subset)
    block_results.append({'block_fraction': fraction, **result})

full_block_auc = block_results[-1]['auc']
recommended_block_fraction = next(
    row['block_fraction'] for row in block_results
    if row['auc'] >= full_block_auc - AUC_TOLERANCE)

sk_model, sk_probability, sk_metrics = fit_score(capped_train)
sk_fpr, sk_tpr, sk_thresholds = roc_curve(test.y, sk_probability)
sk_threshold = float(sk_thresholds[np.argmax(sk_tpr - sk_fpr)])
print('Selected points per block:', selected_cap)
print('Coverage plateau:', f'{recommended_block_fraction:.0%} of training blocks')
print('Full-coverage sklearn AUC:', f"{sk_metrics['auc']:.3f}")

In [ ]:
point_frame = pd.DataFrame(point_results)
block_frame = pd.DataFrame(block_results)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
point_labels = point_frame.points_per_block.astype(str)
axes[0].plot(point_labels, point_frame.auc, 'o-', label='AUC')
axes[0].plot(point_labels, point_frame.accuracy, 'o-', label='Accuracy')
axes[1].plot(block_frame.block_fraction * 100, block_frame.auc, 'o-', label='AUC')
axes[1].plot(block_frame.block_fraction * 100, block_frame.accuracy, 'o-',
             label='Accuracy')
axes[0].set(title='Points within each training block', xlabel='Points per block')
axes[1].set(title='Training-block coverage', xlabel='Training blocks used (%)')
for ax in axes:
    ax.set_ylabel('Held-out score')
    ax.grid(alpha=.25); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Validate and save the Earth Engine random forest

In [ ]:
selected_train_ids = capped_train['_row'].astype(int).tolist()
gee_train = reference_samples.filter(ee.Filter.inList('_row', selected_train_ids))
gee_test = reference_samples.filter(ee.Filter.inList('block_id', sorted(test_blocks)))

def gee_forest():
    return ee.Classifier.smileRandomForest(
        numberOfTrees=N_TREES, variablesPerSplit=VARS_PER_SPLIT,
        minLeafPopulation=MIN_LEAF_POPULATION,
        bagFraction=GEE_BAG_FRACTION, seed=SEED)

gee_holdout_model = gee_forest().setOutputMode('PROBABILITY').train(
    gee_train, 'y', BANDS)
gee_scored = ee.data.computeFeatures({
    'expression': gee_test.classify(gee_holdout_model).select(['y', 'classification']),
    'fileFormat': 'PANDAS_DATAFRAME'})
gee_y = gee_scored.y.astype(int).to_numpy()
gee_probability = gee_scored.classification.to_numpy()
gee_fpr, gee_tpr, gee_thresholds = roc_curve(gee_y, gee_probability)
gee_threshold = float(gee_thresholds[np.argmax(gee_tpr - gee_fpr)])
gee_auc = roc_auc_score(gee_y, gee_probability)
gee_accuracy = accuracy_score(gee_y, gee_probability >= .5)
print(f'Earth Engine held-out AUC {gee_auc:.3f}; accuracy {gee_accuracy:.3f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], '--', color='0.7')
ax.plot(sk_fpr, sk_tpr, label=f'sklearn AUC {sk_metrics["auc"]:.3f}')
ax.plot(gee_fpr, gee_tpr, label=f'Earth Engine AUC {gee_auc:.3f}')
ax.set(xlabel='False positive rate', ylabel='True positive rate',
       title='ROC on held-out blocks', xlim=(0, 1), ylim=(0, 1))
ax.legend(frameon=False); plt.show()

production = reference.copy().assign(
    _order=np.random.default_rng(SEED).random(len(reference)))
production = cap_per_block(production, selected_cap)
production_ids = production['_row'].astype(int).tolist()
gee_production = reference_samples.filter(ee.Filter.inList('_row', production_ids))
final_gee_model = gee_forest().train(
    gee_production, 'y', BANDS)
task = ee.batch.Export.classifier.toAsset(
    final_gee_model, f'{RUN_NAME}_random_forest', MODEL_ASSET)
task.start(); wait(task); wait_for_asset(MODEL_ASSET)
print('Saved model:', MODEL_ASSET)

## Export the viewer report

In [ ]:
import json

def roc_data(fpr, tpr, thresholds):
    return {'false_positive_rate': fpr.tolist(),
            'true_positive_rate': tpr.tolist(),
            'threshold': [float(x) if np.isfinite(x) else None for x in thresholds]}

split_data = reference[['lon', 'lat', 'block_id']].copy()
split_data['split'] = np.where(split_data.block_id.isin(test_blocks),
                               'holdout', 'train')
sk_pred = sk_probability >= .5
gee_pred = gee_probability >= .5
sk_tn, sk_fp, sk_fn, sk_tp = confusion_matrix(test.y, sk_pred).ravel()
gee_tn, gee_fp, gee_fn, gee_tp = confusion_matrix(gee_y, gee_pred).ravel()

report = {
    'schema_version': '1.0',
    'run': {'name': RUN_NAME, 'reference_year': REFERENCE_YEAR,
            'seed': SEED,
            'target_column': TARGET_COLUMN,
            'target_threshold': TARGET_THRESHOLD,
            'block_size_m': BLOCK_SIZE_M},
    'assets': {'points': POINT_ASSET, 'reference_samples': SAMPLE_ASSET,
               'classifier': MODEL_ASSET},
    'input': {'n_observations': len(reference),
              'n_positive': int(reference.y.sum()),
              'positive_fraction': float(reference.y.mean()),
              'n_blocks': int(reference.block_id.nunique())},
    'split': {'n_training_points': len(train), 'n_test_points': len(test),
              'n_training_blocks': int(train.block_id.nunique()),
              'n_test_blocks': int(test.block_id.nunique()),
              'points': split_data.to_dict('records')},
    'experiments': {'points_per_block': point_results,
                    'block_fraction': block_results},
    'selection': {'auc_tolerance': AUC_TOLERANCE,
                  'points_per_block': selected_cap,
                  'recommended_minimum_block_fraction': recommended_block_fraction,
                  'production_uses_all_blocks': True,
                  'production_points': int(len(production)),
                  'production_blocks': int(production.block_id.nunique())},
    'sklearn': {'parameters': {'number_of_trees': N_TREES,
                               'max_features': 'sqrt', 'bootstrap': True,
                               'seed': SEED},
                'auc': sk_metrics['auc'], 'accuracy': sk_metrics['accuracy'],
                'threshold': sk_threshold,
                'confusion_matrix': {'tn': int(sk_tn), 'fp': int(sk_fp),
                                     'fn': int(sk_fn), 'tp': int(sk_tp)},
                'roc': roc_data(sk_fpr, sk_tpr, sk_thresholds)},
    'earth_engine': {'parameters': {'number_of_trees': N_TREES,
                                    'variables_per_split': VARS_PER_SPLIT,
                                    'minimum_leaf_population': MIN_LEAF_POPULATION,
                                    'bag_fraction': GEE_BAG_FRACTION,
                                    'seed': SEED},
                     'auc': float(gee_auc), 'accuracy': float(gee_accuracy),
                     'threshold': gee_threshold,
                     'confusion_matrix': {'tn': int(gee_tn), 'fp': int(gee_fp),
                                          'fn': int(gee_fn), 'tp': int(gee_tp)},
                     'roc': roc_data(gee_fpr, gee_tpr, gee_thresholds)}
}
with open(f'{RUN_NAME}_results.json', 'w') as handle:
    json.dump(report, handle, indent=2,
              default=lambda value: value.item() if isinstance(value, np.generic) else str(value))
files.download(f'{RUN_NAME}_results.json')